# 53 · Zone-anchored landmarks: a general reconstruction without external atlases

Notebook 51 took landmarks and gauge from external kidney segment atlases. A general method for a
continuous repeated structure cannot assume such atlases exist. Here landmarks come from the data's
own coarse discrete zones, the reviewed S1/S2/S3 labels, standing in for any tissue's clusters.

**Cross-fitting.** Landmark selection is cross-fitted on independent read halves:
- landmarks are selected on one half by the zone F statistic, with the scored gene fold excluded;
- positions are fitted on the other half, and then the roles swap;
- the two half posteriors multiply.

**Within zones,** positions are refined continuously, with a soft zone prior (ε = 0.10).

**External atlases are used only for evaluation.** This frees microdissection, female mouse snRNA
and human half A as extra references.

**Configuration.** The model configuration is the one notebook 52 adopted, read from its
`summary.json`. If notebook 52 adopted "LRS with uncertainty", the zone arm is the cross-fitted
ratio LRS-Z.

**Placebo.** Expression-matched genes from the lowest-F genes.

**Rules.** Keep or abandon against LCP-ext, as frozen in `SCRATCH/ws7/ledger_entry_phase3.md`.
Kidney data only; no tissue coordinates. Cohort: two male control mice and two cortex sections from
one male human donor. Everything is descriptive.

## 1 · Folders and fixed settings

In [ ]:
import argparse
import json
import math
import os
import sys
from concurrent.futures import ProcessPoolExecutor
from hashlib import sha256
from multiprocessing import get_context
from pathlib import Path

start = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
project = next(p for p in (start, *start.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
upstream13 = results_root / 'minimal_pt_scfates'
upstream36 = results_root / 'pt_reconstruction_v2' / 'nb36'
external = data_root / 'external'
output_root = results_root / 'pt_reconstruction_v3'
output = output_root / 'nb53'
upstream51 = output_root / 'nb51'
output.mkdir(parents=True, exist_ok=True)
SAMPLES = {'mouse': ['Ctrl1A2', 'Ctrl1A4'], 'human': ['HUK1_COR1', 'HUK1_MED1']}
source_paths = {n: data_root / 'tubule_by_gene' / f'{n}_tubule_by_gene_caleb.h5ad'
                for names in SAMPLES.values() for n in names}
required = [upstream13 / 'cross_species_pt_scfates.h5ad', upstream13 / 'cross_species_nephron_scfates.h5ad',
            upstream13 / 'ortholog_map_used.csv', upstream36 / 'gene_sets_and_external_contrasts.csv',
            upstream36 / 'p4_heldout_prediction.csv',
            results_root / 'pt_reconstruction_v2' / 'nb35' / 'p2_absorption.csv',
            external / 'segment_microdissection' / 'GSE150338_Mouse_TPM_Replicates.txt.gz',
            external / 'segment_microdissection' / 'GSE56743_RPKM.without.PAseq.txt.gz',
            external / 'census_pt_segments' / 'mouse_pt_segment_pseudobulk_counts.csv.gz',
            external / 'census_pt_segments' / 'human_pt_segment_pseudobulk_counts.csv.gz',
            upstream51 / 'landmark_panel.csv', output_root / 'nb52' / 'summary.json', *source_paths.values()]
for path in required:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path}')

NOTEBOOK_LOGIC_VERSION = '53.zone_landmarks.1'  # Bump when a cached calculation changes.
EARLY_MARKERS, LATE_MARKERS = ('Slc5a2', 'Slc5a12', 'Gatm'), ('Slc22a7', 'Slc7a13', 'Cyp7b1')
N_FOLDS, MIN_DETECTION, ANCHOR_LOOSE = 3, .10, .5   # notebook 36
LANDMARK_THRESHOLD, LANDMARK_MIN_LEVEL = 1.0, 1.0
EQUAL_DEPTH_QUANTILE, SPLIT_EPSILON, SPLIT_SEED = .20, .5, 35   # notebook 36
MOUSE_SHAPE_POINTS, HUMAN_SHAPE_POINTS = (1 / 6, 1 / 2, 5 / 6), (1 / 3, 5 / 6)
INJECTION_SHARE_MOD, INJECTION_LOG_AMPLITUDE, OWN_INJECTION_SEED = 10, math.log(2), 357
BASIS_DF, P4C_DF = 6, 4
FIT_SETTINGS = dict(df=6, lam=10.0, coarse=50, fine=200, coarse_max=300, fine_max=500, tol_mean=1e-4,
                    tol_rel=1e-9, patience=5)
PLACEBO_MAX_CONTRAST, PLACEBO_SEED, STRESS_SHARE, STRESS_SEED = .2, 521, .7, 52
LRS_BOOT, LRS_BOOT_SEED, WSR_PERMUTATIONS, WSR_SEED, WSR_ALPHA = 200, 520, 500, 61, .01
CURVE_POINTS, CURVE_SHIFT, CONSERVED_MIN = 21, .25, .5
WORKERS = int(os.environ.get('PSEUDOSPACE_53_WORKERS', '36'))
print('Results folder:', output)

## 2 · Inputs: notebook 13's structures and the original counts (as notebook 36)

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import sparse
from scipy.stats import gaussian_kde, spearmanr

import pseudospace.landmark_reconstruction as rec_module
from pseudospace.anchor_coordinate import (bulk_segment_profiles, census_segment_profiles, conserved_anchor_mask,
                                           positive_control_masks, pt_specific_mask, segment_contrast)
from pseudospace.coordinate_arms import heldout_prediction, registration_gap, within_segment_agreement
from pseudospace.coordinate_evaluation import (absorption_ratio, binomial_thin_to_depth, concordance,
                                               group_rank_correlations, log_normalize, poisson_count_split,
                                               scale_counts, sha256_folds)
from pseudospace.landmark_position import landmark_ratio_score, project_lcp, residual_spline_gain, scaled_gain
from pseudospace.landmark_reconstruction import (combine_half_posteriors, deviance_gain_repaired,
                                                 design_effect_temperature, fit_landmark_model, lrs_bootstrap_sd,
                                                 project_landmark_model, standardized_difference, wsr_correlations,
                                                 wsr_statistic, zone_f_statistic, zone_log_prior, zone_order)
from pseudospace.pathway_inputs import rebuild_pt_expression
from pseudospace.stage_cache import cached_payload, code_digest, digest, stage_is_fresh, stage_key
from pseudospace.within_segment_validation import fit_segment_covariate_oracle, predict_segment_covariate_oracle

saved_pt = ad.read_h5ad(upstream13 / 'cross_species_pt_scfates.h5ad', backed='r').obs.copy()
saved_neph = ad.read_h5ad(upstream13 / 'cross_species_nephron_scfates.h5ad', backed='r').obs.copy()
key_files = [p for p in sorted((upstream13 / 'stage_cache').glob('pass2_nephron_embedding__*.h5ad.key.json'))
             if "'logic': '13-nephron-pass2-v1'" in json.loads(p.read_text())['params']]
if len(key_files) != 1:
    raise FileNotFoundError(f'Expected one notebook-13 pass-2 cache, found {len(key_files)}; rerun notebook 13.')
neph = ad.read_h5ad(str(key_files[0])[:-len('.key.json')])
if not neph.obs_names.equals(saved_neph.index):
    raise ValueError('Notebook 13 pass-2 cache does not match its saved nephron structures.')
pt_rows = np.flatnonzero(neph.obs.coarse_class.astype(str).eq('PT').to_numpy())
if not neph.obs_names[pt_rows].equals(saved_pt.index):
    raise ValueError('PT rows differ from notebook 13 saved PT structures.')
rebuilt = rebuild_pt_expression(
    saved_neph[['sample', 'comparison_species', 'shared_pseudospace']].astype({'sample': str, 'comparison_species': str}),
    source_paths, pd.read_csv(upstream13 / 'ortholog_map_used.csv'), target_sum=1e4)
COUNTS = sparse.csr_matrix(rebuilt.layers['counts']).astype(np.int64)
genes_all = pd.Index(rebuilt.var_names)
measured_both = rebuilt.var.measured_in_both_inputs.to_numpy(bool)
cache_genes = pd.Index(neph.var_names)
cache_cols = genes_all.get_indexer(cache_genes)
if (cache_cols < 0).any() or abs(COUNTS[:, cache_cols] - sparse.csr_matrix(neph.layers['counts'])).max() != 0:
    raise ValueError('Rebuilt counts differ from notebook 13 cached counts.')
CONSTRUCTION_GENES = cache_genes[(neph.var.highly_variable_for_harmony & neph.var.highly_variable).to_numpy(bool)]
neph_obs = neph.obs.copy()
del neph

species_pt = saved_pt.comparison_species.astype(str).to_numpy()
specimen_pt = saved_pt['sample'].astype(str).to_numpy()
segment_pt = saved_pt.segment_class.astype(str).str.replace('PT-', '', regex=False).to_numpy()
human_pt = species_pt == 'human'
SCF_SAVED = saved_pt.shared_pseudospace.to_numpy(float)
DPT_SAVED = saved_pt.dpt_pseudospace.to_numpy(float)
PT_COUNTS = COUNTS[pt_rows]
LIBRARY_PT = np.asarray(PT_COUNTS[:, measured_both].sum(axis=1)).ravel().astype(float)
if (LIBRARY_PT <= 0).any():
    raise ValueError('A PT structure has no counts in the shared panel.')
EVAL_ORIGINAL = log_normalize(PT_COUNTS, LIBRARY_PT)
detection = pd.DataFrame({sp: np.asarray((PT_COUNTS[species_pt == sp] > 0).mean(axis=0)).ravel()
                          for sp in SAMPLES}, index=genes_all)
print(f'{len(pt_rows):,} PT structures · {len(genes_all):,} genes · counts reproduce notebook 13')

## 3 · External references, notebook 36's evaluation sets and the landmark panels

- **Selection references** (landmarks only): mouse microdissection, female mouse snRNA, human snRNA
  donor half A.
- **Evaluation references** (metrics only): male mouse snRNA, including its adjacent-segment
  contrasts, and human half B.

A landmark has |S3 − early| ≥ 1 with the same sign in its species' selection references, level ≥ 1,
detection ≥ 10% in that species' PT and PT specificity in that species (notebook 36's rule, per
species). Notebook 36's conserved anchors and evaluation masks are rebuilt unchanged and checked.

In [ ]:
mouse_bulk = pd.read_csv(external / 'segment_microdissection' / 'GSE150338_Mouse_TPM_Replicates.txt.gz', sep='\t')
mouse_micro = bulk_segment_profiles(mouse_bulk.drop_duplicates('Gene_symbol').set_index('Gene_symbol'),
                                    '_PT{seg}_').reindex(genes_all)


def census(label, segment_map, early):
    counts = pd.read_csv(external / 'census_pt_segments' / f'{label}_pt_segment_pseudobulk_counts.csv.gz', index_col=0)
    meta = pd.read_csv(external / 'census_pt_segments' / f'{label}_pt_segment_pseudobulk_meta.csv', index_col=0)
    return census_segment_profiles(counts, meta.loc[counts.index], segment_map, early=early).reindex(genes_all)


mouse_sn = census('mouse', {'epithelial cell of proximal tubule segment 1': 'S1',
                            'epithelial cell of proximal tubule segment 2': 'S2',
                            'epithelial cell of proximal tubule segment 3': 'S3'}, ['S1', 'S2'])
human_sn = census('human', {'kidney proximal convoluted tubule epithelial cell': 'early',
                            'epithelial cell of proximal tubule segment 3': 'S3'}, None)
human_donors = sorted(set(human_sn.columns.get_level_values('donor')))
HALF_A, HALF_B = human_donors[0::2], human_donors[1::2]
if HALF_A != ['21_015', 'HCA_51_ref', 'PRECISE-34'] or HALF_B != ['HCA_29_ref', 'HCA_55_ref', 'PRECISE-36']:
    raise ValueError(f'Human donor halves differ from the pre-registered split: {HALF_A} / {HALF_B}')


def micro_contrast(frame, late, early):
    return frame[late] - frame[early], frame[[late, early]].max(axis=1)


ext = {'select_mouse_micro': micro_contrast(mouse_micro, 'S3', 'early'),
       'select_mouse_sn_female': segment_contrast(mouse_sn, sex='female'),
       'select_human_sn_halfA': segment_contrast(human_sn, donors=HALF_A),
       'mouse_sn_male': segment_contrast(mouse_sn, sex='male'),
       'mouse_sn_male_S2mS1': segment_contrast(mouse_sn, late='S2', early='S1', sex='male'),
       'mouse_sn_male_S3mS1': segment_contrast(mouse_sn, late='S3', early='S1', sex='male'),
       'mouse_sn_male_S3mS2': segment_contrast(mouse_sn, late='S3', early='S2', sex='male'),
       'mouse_micro': micro_contrast(mouse_micro, 'S3', 'early'),
       'mouse_micro_S2mS1': micro_contrast(mouse_micro, 'S2', 'S1'),
       'mouse_micro_S3mS1': micro_contrast(mouse_micro, 'S3', 'S1'),
       'mouse_micro_S3mS2': micro_contrast(mouse_micro, 'S3', 'S2'),
       'human_sn_halfB': segment_contrast(human_sn, donors=HALF_B)}
contrast = pd.DataFrame({k: v[0] for k, v in ext.items()})
level = pd.DataFrame({k: v[1] for k, v in ext.items()})

classes = neph_obs.coarse_class.astype(str).to_numpy()
specimen_neph = neph_obs['sample'].astype(str).to_numpy()
library_neph = np.asarray(COUNTS[:, measured_both].sum(axis=1)).ravel().astype(float)
normalized = sparse.csr_matrix(COUNTS.multiply((1e4 / library_neph)[:, None]))
class_means = {}
for sp, names in SAMPLES.items():
    columns = {}
    for cls in sorted(set(classes)):
        per = [np.asarray(normalized[(classes == cls) & (specimen_neph == n)].mean(axis=0)).ravel()
               for n in names if ((classes == cls) & (specimen_neph == n)).sum() >= 20]
        columns[cls] = np.mean(per, axis=0)
    class_means[sp] = pd.DataFrame(columns, index=genes_all)
del normalized
eligible_anchor = (pd.Series(measured_both, index=genes_all) & detection.mouse.ge(MIN_DETECTION)
                   & detection.human.ge(MIN_DETECTION) & pt_specific_mask(class_means))
selection = ['select_mouse_micro', 'select_mouse_sn_female', 'select_human_sn_halfA']
A_LOOSE = conserved_anchor_mask(contrast[selection], level[selection], eligible_anchor, threshold=ANCHOR_LOOSE)
markers = pd.Series(genes_all.isin(EARLY_MARKERS + LATE_MARKERS), index=genes_all)
saved36 = pd.read_csv(upstream36 / 'gene_sets_and_external_contrasts.csv', index_col=0)
if not (saved36.anchor_loose_A166rule.astype(bool).reindex(genes_all).fillna(False) == A_LOOSE).all():
    raise ValueError("Notebook 36's conserved anchors do not reproduce.")


def evaluation_mask(sp, reference):
    return (pd.Series(measured_both, index=genes_all) & ~A_LOOSE & ~markers & detection[sp].ge(MIN_DETECTION)
            & level[reference].ge(1) & contrast[reference].notna())


control_eligible = (pd.Series(measured_both, index=genes_all) & ~A_LOOSE & ~markers & detection.mouse.ge(MIN_DETECTION)
                    & detection.human.ge(MIN_DETECTION) & level['mouse_sn_male'].ge(1))
D_OPPOSITE, D_FLAT = positive_control_masks(contrast['human_sn_halfB'], contrast['mouse_sn_male'], contrast['mouse_micro'],
                                            level['human_sn_halfB'], control_eligible)
FOLD = pd.Series(sha256_folds(genes_all, N_FOLDS), index=genes_all)
EVAL_GENES = genes_all[(evaluation_mask('mouse', 'mouse_sn_male') | evaluation_mask('human', 'human_sn_halfB')
                        | evaluation_mask('mouse', 'mouse_micro') | D_OPPOSITE | D_FLAT).to_numpy()]
EVAL_X = EVAL_ORIGINAL[:, genes_all.get_indexer(EVAL_GENES)]
fold_of_eval = FOLD.loc[EVAL_GENES].to_numpy()

# Landmark panels, one per species, from selection references only.
LANDMARK_RULES = {'mouse': ['select_mouse_micro', 'select_mouse_sn_female'], 'human': ['select_human_sn_halfA']}
panel_rows, LANDMARKS = [], {}
for sp, cols in LANDMARK_RULES.items():
    eligible = detection[sp].ge(MIN_DETECTION) & pt_specific_mask({sp: class_means[sp]})
    mask = conserved_anchor_mask(contrast[cols], level[cols], eligible, threshold=LANDMARK_THRESHOLD,
                                 min_level=LANDMARK_MIN_LEVEL)
    genes = list(genes_all[mask.to_numpy()])
    LANDMARKS[sp] = genes
    for g in genes:
        panel_rows.append({'species': sp, 'gene': g, 'sign': int(np.sign(contrast.loc[g, cols[0]])),
                           'fold': int(FOLD[g]), 'contrast_selection': float(contrast.loc[g, cols[0]])})
PANEL = pd.DataFrame(panel_rows)
PANEL.to_csv(output / 'landmark_panel.csv', index=False)
if not PANEL[['species', 'gene', 'sign']].equals(pd.read_csv(upstream51 / 'landmark_panel.csv')[['species', 'gene', 'sign']]):
    raise ValueError("The external landmark panels differ from notebook 51's.")
half_a = human_sn.loc[:, human_sn.columns.get_level_values('donor').isin(HALF_A)]
SHAPE_POINTS = {'mouse': np.array(MOUSE_SHAPE_POINTS), 'human': np.array(HUMAN_SHAPE_POINTS)}
SHAPE_SOURCE = {'mouse': mouse_micro[['S1', 'S2', 'S3']] * math.log(2),
                'human': pd.concat([half_a.xs('early', axis=1, level='segment').mean(axis=1),
                                    half_a.xs('S3', axis=1, level='segment').mean(axis=1)], axis=1) * math.log(2)}


def shape_values(sp, genes):
    """External segment-level log profile of each gene (selection references), genes x shape points."""
    values = SHAPE_SOURCE[sp].loc[list(genes)].to_numpy(float)
    if not np.isfinite(values).all():
        raise ValueError('Missing external profile for a panel gene.')
    return values


display(PANEL.groupby(['species', 'sign']).size().unstack().rename(columns={-1: 'early-high', 1: 'late-high'}))
print('Evaluation genes:', len(EVAL_GENES), '· landmarks also in the evaluation set:',
      {sp: int(pd.Index(g).isin(EVAL_GENES).sum()) for sp, g in LANDMARKS.items()})

## 4 · Comparators: SCF13 and DPT13 from notebook 36's cache, unchanged

The full fit must reproduce notebook 13's saved coordinates exactly before any refit is used.

In [ ]:
def load36(stage, spec=None):
    hits = []
    for path in upstream36.joinpath('stage_cache').glob(f'{stage}__*.npz'):
        params = json.loads(Path(str(path) + '.key.json').read_text())['params']
        if spec is None or f"'spec': '{spec}'" in params:
            hits.append(path)
    if len(hits) != 1:
        raise FileNotFoundError(f'Expected one notebook-36 cache for {stage} {spec}, found {len(hits)}.')
    with np.load(hits[0], allow_pickle=False) as stored:
        return {k: stored[k] for k in stored.files}


SPECS = {'full': 'unperturbed', 'fold0': 'fold 0', 'fold1': 'fold 1', 'fold2': 'fold 2', 'ed': 'equal depth 352',
         'half_A': 'split A', 'half_B': 'split B'}
STAGES = {'full': 'arm_scf13_full', 'fold0': 'arm_scf13_fold0', 'fold1': 'arm_scf13_fold1', 'fold2': 'arm_scf13_fold2',
          'ed': 'arm_scf13_ed', 'half_A': 'arm_scf13_half_A', 'half_B': 'arm_scf13_half_B'}
fits36 = {key: load36(STAGES[key], SPECS[key]) for key in SPECS}
if np.abs(fits36['full']['scfates'] - SCF_SAVED).max() > 1e-9 or np.nanmax(np.abs(fits36['full']['dpt'] - DPT_SAVED)) > 1e-6:
    raise ValueError("Notebook 36's cached SCF13/DPT13 do not reproduce notebook 13.")
COORD = {'SCF13': {key: fit['scfates'] for key, fit in fits36.items()},
         'DPT13': {key: fit['dpt'] for key, fit in fits36.items()}}
COORD['SCF13']['loso'] = load36('loso_SCF13')['loso']
print('SCF13 and DPT13 refits loaded and reproduced.')

## 5 · Perturbed counts shared by every arm (notebook 36's seeds) and the LRS comparator

LRS is log((late + 0.5)/(early + 0.5)) over the species' landmark counts. A fold version drops that
fold's landmarks. For metrics that need a [0, 1] scale (splines, gaps), it is mapped to the
within-species pooled rank. That is monotone, so every Spearman metric is unchanged.

In [ ]:
def equal_depth(counts, seed):
    library = np.asarray(counts[:, measured_both].sum(axis=1)).ravel().astype(float)
    target = float(np.quantile(library[pt_rows], EQUAL_DEPTH_QUANTILE))
    return binomial_thin_to_depth(counts, library, target, np.random.default_rng(seed))


ED_352 = equal_depth(COUNTS, 352)
SPLIT_A, SPLIT_B = poisson_count_split(COUNTS, SPLIT_EPSILON, np.random.default_rng(SPLIT_SEED))


def hash_int(gene):
    return int(sha256(str(gene).encode()).hexdigest(), 16)


NONANCHOR_INJECTED = sorted([g for g in CONSTRUCTION_GENES if not A_LOOSE.get(g, False)
                             and g not in EARLY_MARKERS + LATE_MARKERS and hash_int(g) % INJECTION_SHARE_MOD == 0], key=hash_int)


def inject(counts, genes, s0, seed):
    """Human-only exp(±log 2 (2 s0 − 1)) on the listed genes; signs alternate in hash order (notebook 36)."""
    genes = sorted(genes, key=hash_int)
    signs = np.where(np.arange(len(genes)) % 2 == 0, 1.0, -1.0)
    f = INJECTION_LOG_AMPLITUDE * (2 * s0[human_pt] - 1)
    return scale_counts(counts, pt_rows[human_pt], genes_all.get_indexer(genes), f[:, None] * signs[None, :],
                        np.random.default_rng(seed)), genes


INJECTED_36, _ = inject(COUNTS, NONANCHOR_INJECTED, SCF_SAVED, 351)
VARIANTS = {'full': COUNTS, 'ed': ED_352, 'half_A': SPLIT_A, 'half_B': SPLIT_B}


def species_block(counts, sp, genes):
    rows = pt_rows[species_pt == sp]
    block = sparse.csr_matrix(counts[rows][:, genes_all.get_indexer(genes)])
    library = np.asarray(counts[rows][:, measured_both].sum(axis=1)).ravel().astype(float)
    return block, library


def fold_landmarks(sp, k=None):
    return [g for g in LANDMARKS[sp] if k is None or FOLD[g] != k]


def lrs(counts, k=None):
    out = np.full(len(pt_rows), np.nan)
    for sp in SAMPLES:
        genes = fold_landmarks(sp, k)
        sign = PANEL.set_index(['species', 'gene']).loc[sp].sign.loc[genes].to_numpy()
        block, _ = species_block(counts, sp, genes)
        late = np.asarray(block[:, sign > 0].sum(axis=1)).ravel()
        early = np.asarray(block[:, sign < 0].sum(axis=1)).ravel()
        score = landmark_ratio_score(late, early)
        out[species_pt == sp] = (pd.Series(score).rank().to_numpy() - .5) / len(score)
    return out


COORD['LRS'] = {'full': lrs(COUNTS), **{f'fold{k}': lrs(COUNTS, k) for k in range(N_FOLDS)},
                'ed': lrs(ED_352), 'half_A': lrs(SPLIT_A), 'half_B': lrs(SPLIT_B)}
print('LRS built for the full data, three folds, equal depth and both count-split halves.')

## 6 · Comparator LCP (notebook 51's cached fits) and notebook 52's adopted configuration

In [ ]:
def load51(stage):
    hits = list(upstream51.joinpath('stage_cache').glob(f'{stage}__*.npz'))
    if len(hits) != 1:
        raise FileNotFoundError(f'Expected one notebook-51 cache for {stage}, found {len(hits)}.')
    with np.load(hits[0], allow_pickle=False) as stored:
        return {k: stored[k] for k in stored.files}


COORD['LCP'] = {}
for key in ('full', 'fold0', 'fold1', 'fold2', 'ed', 'half_A', 'half_B'):
    z = np.full(len(pt_rows), np.nan)
    for sp in SAMPLES:
        z[species_pt == sp] = load51(f'lcp_{sp}_{key}')['mean']
    COORD['LCP'][key] = z
loso = np.full(len(pt_rows), np.nan)
for sp in SAMPLES:
    spec = specimen_pt[species_pt == sp]
    block, library = species_block(COUNTS, sp, LANDMARKS[sp])
    for name in SAMPLES[sp]:
        fit = load51(f'lcp_{sp}_loso_train_without_{name}')
        model = {'eta': fit['eta'], 'theta': fit['theta'], 'temperature': float(fit['temperature']), 'grid': fit['grid']}
        rows = spec == name
        loso[np.flatnonzero(species_pt == sp)[rows]] = project_lcp(model, block[rows].toarray(), library[rows])['mean']
COORD['LCP']['loso'] = loso
decision52 = json.loads((output_root / 'nb52' / 'summary.json').read_text())['decision']
ADOPTED = decision52['adopted configuration']
ARM_CFG = {'M0': dict(basis='linear', balanced=False), 'M1': dict(basis='spline', balanced=False),
           'M1B': dict(basis='spline', balanced=True), 'M1BC': dict(basis='spline', balanced=True),
           'M1BCJ': dict(basis='spline', balanced=True)}
MODEL = ADOPTED in ARM_CFG
CALIBRATED = ADOPTED in ('M1BC', 'M1BCJ')
if ADOPTED == 'M1BCJ':
    print('Notebook 52 adopted the joint species fit. Zone landmarks are fitted per species here (no shared '
          'shapes), and this is recorded as a deviation.')
print('Notebook 52 adopted configuration:', ADOPTED, '· model fits in this notebook:', MODEL)

## 7 · Zone-anchored landmarks (ZAL)

**Zones** are the reviewed S1/S2/S3 labels. Their order comes from the data: the shortest
Hamiltonian path through the specimen-balanced zone-mean log-rate profiles on read half A, starting
at S1. Guard: the path must be S1–S2–S3 in both species.

**Landmarks** are chosen on one read half:
- eligible genes are detected in ≥ 10% of the species' PT and PT-specific;
- genes are ranked by quasi-Poisson zone F;
- the top 500 are kept, excluding the scored fold.

**Shapes** are log zone rates placed at zone centres, with zone widths equal to the
specimen-balanced occupancy fractions on the same half.

**Fitting.** Positions are fitted on the other half, with the zone prior (ε = 0.10), and then
swapped. The posterior is the product of the two half posteriors divided by the prior once.

**Arms.**
- ZAL;
- ZAL-NP, without the zone prior;
- ZAL-PL, the placebo: 500 expression-matched genes from the 2,000 lowest-F eligible genes;
- LRS-Z, the landmark ratio over the zone landmarks, cross-fitted the same way and averaged on the
  rank gauge.

In [ ]:
ZONE = pd.Series(segment_pt).map({'S1': 0, 'S2': 1, 'S3': 2}).to_numpy()
ZONE_SEED, N_LANDMARKS, ZONE_EPS, PLACEBO_POOL = 53, 500, .10, 2000
ELIGIBLE = {sp: (detection[sp].ge(MIN_DETECTION) & pt_specific_mask({sp: class_means[sp]})).to_numpy() for sp in SAMPLES}


def halves(counts):
    return poisson_count_split(counts, .5, np.random.default_rng(ZONE_SEED))


def select(counts_half, sp, rows=None):
    """Zone statistics on one read half for one species (optionally a row subset)."""
    idx = np.flatnonzero(species_pt == sp)
    if rows is not None:
        idx = idx[rows]
    genes = genes_all[ELIGIBLE[sp]]
    block = sparse.csr_matrix(counts_half[pt_rows[idx]][:, np.flatnonzero(ELIGIBLE[sp])])
    library = np.asarray(counts_half[pt_rows[idx]][:, measured_both].sum(axis=1)).ravel().astype(float)
    zone, spec = ZONE[idx], specimen_pt[idx]
    F, rates = zone_f_statistic(block, library, zone, spec, 3)
    profiles = np.log(rates.T + 1e-7)
    order, _ = zone_order(profiles, start=0)
    fractions = np.mean([[np.mean(zone[spec == s] == j) for j in range(3)] for s in np.unique(spec)], axis=0)
    edges = np.r_[0, np.cumsum(fractions)]
    edges[-1] = 1.0
    totals = np.asarray(block.sum(axis=0)).ravel()
    return {'F': pd.Series(F, index=genes), 'rates': pd.DataFrame(rates, index=genes), 'order': order,
            'edges': edges, 'totals': pd.Series(totals, index=genes)}


def landmarks_from(sel, k=None, placebo=False):
    F = sel['F'] if k is None else sel['F'][FOLD.loc[sel['F'].index].to_numpy() != k]
    top = list(F.sort_values(ascending=False).index[:N_LANDMARKS])
    if not placebo:
        return top
    pool = list(F.sort_values().index[:PLACEBO_POOL])
    log_total = np.log1p(sel['totals'].loc[pool].to_numpy())
    free, chosen = np.ones(len(pool), bool), []
    for g in top:
        gap = np.where(free, np.abs(log_total - np.log1p(sel['totals'][g])), np.inf)
        best = int(np.argmin(gap))
        free[best] = False
        chosen.append(pool[best])
    return chosen


def zone_block(counts_half, sp, sel, genes, *, rows=None, prior=True, temperature=1.0):
    idx = np.flatnonzero(species_pt == sp)
    if rows is not None:
        idx = idx[rows]
    block = sparse.csr_matrix(counts_half[pt_rows[idx]][:, genes_all.get_indexer(genes)])
    library = np.asarray(counts_half[pt_rows[idx]][:, measured_both].sum(axis=1)).ravel().astype(float)
    centres = (sel['edges'][:-1] + sel['edges'][1:]) / 2
    out = {'counts': block.toarray().astype(np.int64), 'library': library, 'specimen': specimen_pt[idx],
           'shape_points': centres, 'shape_values': np.log(sel['rates'].loc[genes].to_numpy() + 1e-7),
           'temperature': float(temperature)}
    if prior:
        out.update(labels=ZONE[idx], zone_edges=sel['edges'], eps=ZONE_EPS)
    return out


# Zone-order guard and selections on the full data.
COUNTS_A, COUNTS_B = halves(COUNTS)
SEL = {(sp, h): select(c, sp) for sp in SAMPLES for h, c in (('A', COUNTS_A), ('B', COUNTS_B))}
order_rows = [{'species': sp, 'half': h, 'zone order': '-'.join('S%d' % (j + 1) for j in s['order']),
               'edges': np.round(s['edges'], 3).tolist()} for (sp, h), s in SEL.items()]
display(pd.DataFrame(order_rows))
if any(s['order'] != [0, 1, 2] for s in SEL.values()):
    raise RuntimeError('Inferred zone order is not S1-S2-S3: stop and report (pre-registered guard).')
overlap = {sp: len(set(landmarks_from(SEL[(sp, 'A')])) & set(LANDMARKS[sp])) for sp in SAMPLES}
print('Zone landmarks shared with the external panels (half A selection):', overlap)
pd.DataFrame([{'species': sp, 'half': h, 'gene': g, 'F': float(s['F'][g])} for (sp, h), s in SEL.items()
              for g in landmarks_from(s)]).to_csv(output / 'zone_landmarks.csv', index=False)

### ZAL fits (cross-fitted halves) and the LRS-Z ratio

**Variants per species:**
- full;
- three gene folds;
- equal depth: the same pipeline on equal-depth counts, split with seed 53;
- LOSO for each specimen: select and fit on the other specimen, then project the held-out one;
- human own-landmark injection, for ZAL and ZAL-NP.

Each variant is two half fits:
- AB: select on A, fit on B;
- BA: select on B, fit on A.

In [ ]:
cache_root = output / 'stage_cache'
code = digest([NOTEBOOK_LOGIC_VERSION, Path(rec_module.__file__), project / 'pseudospace' / 'landmark_position.py'])
progress_log = output / 'progress.log'
FIT_KW = {**ARM_CFG.get(ADOPTED, ARM_CFG['M1B']), **FIT_SETTINGS}
ZAL_ARMS = ['ZAL', 'ZAL-NP', 'ZAL-PL'] if MODEL else []
ED_A, ED_B = halves(ED_352)
SEL_ED = {(sp, h): select(c, sp) for sp in SAMPLES for h, c in (('A', ED_A), ('B', ED_B))}
SEL_LOSO = {}
for sp in SAMPLES:
    spec = specimen_pt[species_pt == sp]
    for name in SAMPLES[sp]:
        for h, c in (('A', COUNTS_A), ('B', COUNTS_B)):
            SEL_LOSO[(sp, name, h)] = select(c, sp, rows=spec != name)


def _run(job):
    from threadpoolctl import threadpool_limits
    with threadpool_limits(1):
        (res,), diag = fit_landmark_model([job['block']], **job['kwargs'])
    payload = {k: np.asarray(v) for k, v in diag.items()}
    for key in ('mean', 'sd', 'eta', 'theta', 'grid'):
        payload[key] = np.asarray(res[key])
    payload['posterior'] = res['posterior'].astype(np.float32)
    payload['temperature'] = np.asarray(res['temperature'])
    return payload


FITS = {}


def run_jobs(jobs):
    """jobs: key -> block. Cached per key and input; results go into FITS."""
    from concurrent.futures import as_completed
    specs, missing = {}, []
    for name, block in jobs.items():
        kwargs = dict(FIT_KW)
        params = dict(kwargs, temperature=block['temperature'], prior='labels' in block)
        inputs = {k: block[k] for k in ('counts', 'library', 'specimen', 'shape_values', 'shape_points')}
        if 'labels' in block:
            inputs.update(labels=block['labels'], edges=block['zone_edges'])
        key = stage_key(f'zal_{name}', params=params, inputs=inputs, code=code_digest(code))
        specs[name] = (params, inputs, {'block': block, 'kwargs': kwargs})
        if not stage_is_fresh(cache_root / f'zal_{name}__{key}.npz', key):
            missing.append(name)
    if missing:
        with ProcessPoolExecutor(max_workers=min(WORKERS, len(missing)), mp_context=get_context('fork')) as pool:
            futures = {pool.submit(_run, specs[m][2]): m for m in missing}
            for future in as_completed(futures):
                name = futures[future]
                result = future.result()
                params, inputs, _ = specs[name]
                cached_payload(f'zal_{name}', lambda result=result: result, root=cache_root, params=params,
                               inputs=inputs, code=code, verbose=False)
                with open(progress_log, 'a') as handle:
                    handle.write(f'{name} done\n')
    for name in jobs:
        params, inputs, _ = specs[name]
        FITS[name] = cached_payload(f'zal_{name}', lambda: None, root=cache_root, params=params, inputs=inputs,
                                    code=code, verbose=False)


def variant_jobs(arm, sp, variant, counts_a, counts_b, sel_a, sel_b, *, k=None, rows=None, temperature=1.0):
    placebo, prior = arm == 'ZAL-PL', arm != 'ZAL-NP'
    genes_a, genes_b = landmarks_from(sel_a, k, placebo), landmarks_from(sel_b, k, placebo)
    return {f'{arm}|{sp}|{variant}|AB': zone_block(counts_b, sp, sel_a, genes_a, rows=rows, prior=prior, temperature=temperature),
            f'{arm}|{sp}|{variant}|BA': zone_block(counts_a, sp, sel_b, genes_b, rows=rows, prior=prior, temperature=temperature)}


def all_jobs(arm, temps):
    jobs = {}
    for sp in SAMPLES:
        T = temps[sp]
        sa, sb = SEL[(sp, 'A')], SEL[(sp, 'B')]
        jobs.update(variant_jobs(arm, sp, 'full', COUNTS_A, COUNTS_B, sa, sb, temperature=T))
        for k in range(N_FOLDS):
            jobs.update(variant_jobs(arm, sp, f'fold{k}', COUNTS_A, COUNTS_B, sa, sb, k=k, temperature=T))
        jobs.update(variant_jobs(arm, sp, 'ed', ED_A, ED_B, SEL_ED[(sp, 'A')], SEL_ED[(sp, 'B')], temperature=T))
        spec = specimen_pt[species_pt == sp]
        for name in SAMPLES[sp]:
            jobs.update(variant_jobs(arm, sp, f'loso_wo_{name}', COUNTS_A, COUNTS_B, SEL_LOSO[(sp, name, 'A')],
                                     SEL_LOSO[(sp, name, 'B')], rows=spec != name, temperature=T))
    return jobs


TEMPERATURE = {sp: 1.0 for sp in SAMPLES}
if MODEL and CALIBRATED:
    # Design-effect temperature for ZAL: two landmark gene halves of the full AB fit, at T = 1.
    jobs = {}
    for sp in SAMPLES:
        genes = landmarks_from(SEL[(sp, 'A')])
        for h in (0, 1):
            jobs[f'ZAL|{sp}|genehalf{h}|AB'] = zone_block(COUNTS_B, sp, SEL[(sp, 'A')],
                                                          [g for g in genes if hash_int(g) % 2 == h])
    run_jobs(jobs)
    for sp in SAMPLES:
        a, b = FITS[f'ZAL|{sp}|genehalf0|AB'], FITS[f'ZAL|{sp}|genehalf1|AB']
        TEMPERATURE[sp], _ = design_effect_temperature(a['mean'], a['sd'], b['mean'], b['sd'])
print('ZAL temperatures:', TEMPERATURE)
if MODEL:
    jobs = {}
    for arm in ZAL_ARMS:
        jobs.update(all_jobs(arm, TEMPERATURE))
    run_jobs(jobs)


def combined(arm, sp, variant, *, project_rows=None, sels=None):
    """Product of the AB and BA half posteriors (divided by the zone prior once), as zone units / 3."""
    ab, ba = FITS[f'{arm}|{sp}|{variant}|AB'], FITS[f'{arm}|{sp}|{variant}|BA']
    grid = ab['grid']
    idx = np.flatnonzero(species_pt == sp)
    if sels is not None:
        sel_a, sel_b = sels
    else:
        sel_a = SEL_LOSO[(sp, variant.replace('loso_wo_', ''), 'A')] if variant.startswith('loso') else (
            SEL_ED[(sp, 'A')] if variant == 'ed' else SEL[(sp, 'A')])
        sel_b = SEL_LOSO[(sp, variant.replace('loso_wo_', ''), 'B')] if variant.startswith('loso') else (
            SEL_ED[(sp, 'B')] if variant == 'ed' else SEL[(sp, 'B')])
    edges = (sel_a['edges'] + sel_b['edges']) / 2
    prior_arm = arm != 'ZAL-NP'
    if project_rows is None:
        post_ab, post_ba = ab['posterior'].astype(float), ba['posterior'].astype(float)
        labels = ZONE[idx]
    else:
        placebo = arm == 'ZAL-PL'
        k = None
        out = []
        for fit, sel, counts in ((ab, sel_a, COUNTS_B), (ba, sel_b, COUNTS_A)):
            genes = landmarks_from(sel, k, placebo)
            block = sparse.csr_matrix(counts[pt_rows[idx[project_rows]]][:, genes_all.get_indexer(genes)])
            library = np.asarray(counts[pt_rows[idx[project_rows]]][:, measured_both].sum(axis=1)).ravel().astype(float)
            lp = zone_log_prior(ZONE[idx[project_rows]], sel['edges'], grid, ZONE_EPS) if prior_arm else None
            res = {'eta': fit['eta'], 'theta': fit['theta'], 'temperature': float(fit['temperature']), 'grid': grid}
            out.append(project_landmark_model(res, block.toarray(), library, log_prior=lp)['posterior'])
        post_ab, post_ba = out
        labels = ZONE[idx[project_rows]]
    log_prior = zone_log_prior(labels, edges, grid, ZONE_EPS) if prior_arm else None
    post = combine_half_posteriors(post_ab, post_ba, log_prior)
    z = post @ grid
    zone_units = np.select([z < edges[1], z < edges[2]], [z / edges[1], 1 + (z - edges[1]) / (edges[2] - edges[1])],
                           2 + (z - edges[2]) / (1 - edges[2]))
    half = {'AB': ab['mean'], 'BA': ba['mean']} if project_rows is None else None
    sd = np.sqrt(np.maximum(post @ grid ** 2 - z ** 2, 0))
    return zone_units / 3, sd, half


SD, HALF = {}, {}
for arm in ZAL_ARMS:
    COORD[arm], SD[arm], HALF[arm] = {}, {}, {}
    for variant in ('full', 'fold0', 'fold1', 'fold2', 'ed'):
        z = np.full(len(pt_rows), np.nan)
        s = np.full(len(pt_rows), np.nan)
        for sp in SAMPLES:
            zz, ss, half = combined(arm, sp, variant)
            z[species_pt == sp], s[species_pt == sp] = zz, ss
            if variant == 'full':
                HALF[arm][sp] = half
        COORD[arm][variant], SD[arm][variant] = z, s
    loso = np.full(len(pt_rows), np.nan)
    for sp in SAMPLES:
        spec = specimen_pt[species_pt == sp]
        for name in SAMPLES[sp]:
            zz, _, _ = combined(arm, sp, f'loso_wo_{name}', project_rows=spec == name)
            loso[np.flatnonzero(species_pt == sp)[spec == name]] = zz
    COORD[arm]['loso'] = loso
    # Count split for ZAL: the two half coordinates (AB uses read half B, BA uses read half A).
    for key, h in (('half_A', 'BA'), ('half_B', 'AB')):
        z = np.full(len(pt_rows), np.nan)
        for sp in SAMPLES:
            z[species_pt == sp] = HALF[arm][sp][h]
        COORD[arm][key] = z


def zone_score(counts, sp, sel, k=None, placebo=False):
    """Landmark ratio over zone landmarks of one selection (sign: S3 rate above S1 rate), plus counts."""
    genes = landmarks_from(sel, k, placebo)
    sign = np.sign(sel['rates'].loc[genes, 2] - sel['rates'].loc[genes, 0]).to_numpy()
    block, _ = species_block(counts, sp, genes)
    late, early = block[:, sign > 0].toarray(), block[:, sign <= 0].toarray()
    return landmark_ratio_score(late.sum(axis=1), early.sum(axis=1)), late, early


def zone_units(score, sp):
    """Zone-unit gauge (plan §1 fallback): zone j from the label, within-zone rank of the score -> [j, j + 1)."""
    zone = ZONE[species_pt == sp]
    out = np.empty(len(score))
    for j in range(3):
        m = zone == j
        out[m] = j + (pd.Series(score[m]).rank().to_numpy() - .5) / m.sum()
    return out


def lrs_zone(counts_a, counts_b, sel_a, sel_b, k=None, placebo=False):
    """Cross-fitted zone ratio: AB (landmarks from A, scored on B) and BA, averaged in zone units, / 3."""
    out = np.full(len(pt_rows), np.nan)
    for sp in SAMPLES:
        units = [zone_units(zone_score(counts, sp, sel[sp], k, placebo)[0], sp)
                 for sel, counts in ((sel_a, counts_b), (sel_b, counts_a))]
        out[species_pt == sp] = np.mean(units, axis=0) / 3
    return out


selA = {sp: SEL[(sp, 'A')] for sp in SAMPLES}
selB = {sp: SEL[(sp, 'B')] for sp in SAMPLES}
edA = {sp: SEL_ED[(sp, 'A')] for sp in SAMPLES}
edB = {sp: SEL_ED[(sp, 'B')] for sp in SAMPLES}
LRSZ_SD = {}
for arm, placebo in (('LRS-Z', False), ('LRS-Z-PL', True)):
    COORD[arm] = {'full': lrs_zone(COUNTS_A, COUNTS_B, selA, selB, placebo=placebo),
                  **{f'fold{k}': lrs_zone(COUNTS_A, COUNTS_B, selA, selB, k, placebo) for k in range(N_FOLDS)},
                  'ed': lrs_zone(ED_A, ED_B, edA, edB, placebo=placebo)}
    # Count split for the ratio: each half's coordinate alone (AB uses read half B, BA uses read half A).
    for key, (sel, counts) in (('half_A', (selB, COUNTS_A)), ('half_B', (selA, COUNTS_B))):
        z = np.full(len(pt_rows), np.nan)
        for sp in SAMPLES:
            z[species_pt == sp] = zone_units(zone_score(counts, sp, sel[sp], None, placebo)[0], sp) / 3
        COORD[arm][key] = z
    # Bootstrap uncertainty (LRS-U): Poisson count plus landmark-gene resampling per half (B = 200), converted
    # to the zone-unit gauge with the within-zone density of the score (delta method); halves combined.
    sd_half = {}
    for h, (sel, counts) in (('AB', (selA, COUNTS_B)), ('BA', (selB, COUNTS_A))):
        v = np.full(len(pt_rows), np.nan)
        for sp in SAMPLES:
            score, late, early = zone_score(counts, sp, sel[sp], None, placebo)
            sd_raw = lrs_bootstrap_sd(late, early, n_boot=LRS_BOOT, seed=LRS_BOOT_SEED)
            zone = ZONE[species_pt == sp]
            sd_u = np.empty(len(score))
            for j in range(3):
                m = zone == j
                sd_u[m] = sd_raw[m] * gaussian_kde(score[m])(score[m])
            v[species_pt == sp] = sd_u / 3
        sd_half[h] = v
    LRSZ_SD[arm] = sd_half

# Own-landmark injection (human-only), s0 = the arm's own full position; the pipeline reruns on it.
INJECTED = {}
OWN_INJECTED_GENES = {}
for arm in ['LRS-Z'] + [a for a in ZAL_ARMS if a != 'ZAL-PL']:
    genes = sorted(set(landmarks_from(SEL[('human', 'A')])) | set(landmarks_from(SEL[('human', 'B')])), key=hash_int)
    OWN_INJECTED_GENES[arm] = [g for g in genes if hash_int(g) % INJECTION_SHARE_MOD == 0]
    INJECTED[arm], _ = inject(COUNTS, OWN_INJECTED_GENES[arm], np.nan_to_num(COORD[arm]['full']), OWN_INJECTION_SEED)
INJ_HALVES, INJ_SEL = {}, {}
for arm, counts in INJECTED.items():
    INJ_HALVES[arm] = halves(counts)
    INJ_SEL[arm] = {h: select(c, 'human') for h, c in zip('AB', INJ_HALVES[arm])}
if MODEL:
    jobs = {}
    for arm in [a for a in ZAL_ARMS if a != 'ZAL-PL']:
        ia, ib = INJ_HALVES[arm]
        jobs.update(variant_jobs(arm, 'human', 'injection', ia, ib, INJ_SEL[arm]['A'], INJ_SEL[arm]['B'],
                                 temperature=TEMPERATURE['human']))
    run_jobs(jobs)
ARMS = ['SCF13', 'DPT13', 'LRS', 'LCP', 'LRS-Z', 'LRS-Z-PL'] + ZAL_ARMS
PLACEBO_ARM = 'ZAL-PL' if MODEL else 'LRS-Z-PL'
FIT_DIAG = pd.DataFrame([{'fit': n, 'iterations coarse': int(f['iterations_coarse']), 'iterations fine': int(f['iterations_fine']),
                          'converged': bool(f['converged'])} for n, f in FITS.items()])
FIT_DIAG.to_csv(output / 'fit_diagnostics.csv', index=False)
CAP_SHARE = float((FIT_DIAG['iterations fine'] >= FIT_SETTINGS['fine_max']).mean()) if len(FIT_DIAG) else np.nan
print('Arms:', ARMS, '· fits:', len(FIT_DIAG), '· share at the cap:', CAP_SHARE)

## 9 · The benchmark of notebook 51, for every arm

- P1 and P1w are fold-protected and scored on evaluation references only. Microdissection P1w is
  reported for continuity and flagged as circular for the landmark arms.
- Also: gene-fold agreement, P5, the library correlation, LOSO, count split and registration gap.

In [ ]:
segment_group = np.where(human_pt, np.where(segment_pt == 'S3', 'S3', 'early'), segment_pt)
groups_within = np.char.add(np.char.add(specimen_pt.astype(str), '|'), segment_group.astype(str))
P1W_EVAL = {('mouse', 'S1'): 'mouse_sn_male_S2mS1', ('mouse', 'S2'): 'mouse_sn_male_S3mS1',
            ('mouse', 'S3'): 'mouse_sn_male_S3mS2', ('human', 'early'): 'human_sn_halfB', ('human', 'S3'): 'human_sn_halfB'}
P1W_MICRO = {('mouse', 'S1'): 'mouse_micro_S2mS1', ('mouse', 'S2'): 'mouse_micro_S3mS1', ('mouse', 'S3'): 'mouse_micro_S3mS2'}


def species_r(expression, z, genes):
    keep = np.isfinite(z)
    table = group_rank_correlations(expression[keep], z[keep], specimen_pt[keep])
    table.index = genes
    return pd.DataFrame({sp: table[names].mean(axis=1, skipna=False) for sp, names in SAMPLES.items()})


def within_r(expression, z, genes):
    keep = np.isfinite(z)
    table = group_rank_correlations(expression[keep], z[keep], groups_within[keep])
    table.index = genes
    return table


def protected(arm, function):
    parts = []
    for k in range(N_FOLDS):
        cols = fold_of_eval == k
        parts.append(function(EVAL_X[:, cols], COORD[arm][f'fold{k}'], EVAL_GENES[cols]))
    return pd.concat(parts).loc[EVAL_GENES]


rows, R = [], {}
for arm in ARMS:
    R[arm], RW = protected(arm, species_r), protected(arm, within_r)
    for sp, ref in (('mouse', 'mouse_sn_male'), ('human', 'human_sn_halfB'), ('mouse', 'mouse_micro'),
                    ('mouse', 'select_mouse_sn_female'), ('human', 'select_human_sn_halfA')):
        mask = evaluation_mask(sp, ref).loc[EVAL_GENES].to_numpy()
        rows.append({'metric': 'P1', 'arm': arm, 'species': sp, 'segment': 'all', 'reference': ref,
                     'circular for landmark arms': ref.startswith('select') or ref == 'mouse_micro',
                     'value': concordance(R[arm].loc[mask, sp], contrast.loc[EVAL_GENES[mask], ref])[0]})
    for refs, tag in ((P1W_EVAL, False), (P1W_MICRO, True)):
        for (sp, seg), ref in refs.items():
            mask = evaluation_mask(sp, ref).loc[EVAL_GENES].to_numpy()
            for n in SAMPLES[sp]:
                rows.append({'metric': 'P1w', 'arm': arm, 'species': sp, 'segment': seg, 'specimen': n, 'reference': ref,
                             'circular for landmark arms': tag,
                             'value': concordance(RW.loc[mask, f'{n}|{seg}'], contrast.loc[EVAL_GENES[mask], ref])[0]})


def cell_rows(metric, arm, table, **extra):
    for rec in table.itertuples():
        sp = 'human' if rec.specimen in SAMPLES['human'] else 'mouse'
        rows.append({'metric': metric, 'arm': arm, 'species': sp, 'segment': rec.segment, 'specimen': rec.specimen,
                     'value': rec.rho, **extra})


log_library = np.log(LIBRARY_PT)
for arm in ARMS:
    c = COORD[arm]
    for a, b in ((0, 1), (0, 2), (1, 2)):
        cell_rows('gene-fold agreement', arm, within_segment_agreement(c[f'fold{a}'], c[f'fold{b}'], specimen_pt, segment_pt),
                  pair=f'{a}-{b}')
    cell_rows('P5 equal depth', arm, within_segment_agreement(c['ed'], c['full'], specimen_pt, segment_pt))
    cell_rows('count split A vs B', arm, within_segment_agreement(c['half_A'], c['half_B'], specimen_pt, segment_pt))
    cell_rows('position vs log library', arm, within_segment_agreement(c['full'], log_library, specimen_pt, segment_pt))
    if 'loso' in c:
        cell_rows('LOSO', arm, within_segment_agreement(c['loso'], c['full'], specimen_pt, segment_pt))
    for key in ('full', 'fold0', 'fold1', 'fold2', 'half_A', 'half_B'):
        gap, _ = registration_gap(c[key], species_pt, specimen_pt, segment_pt)
        for seg, value in gap.items():
            rows.append({'metric': 'registration gap (human − mouse median)', 'arm': arm, 'species': 'both',
                         'segment': seg, 'refit': key, 'value': float(value)})

### P2 · absorption of an injected human-only gradient

Each arm's own landmarks are injected, with s0 set to the arm's own full position. LRS is
recomputed. LCP's value is read from notebook 51, and SCF13 and DPT13 from notebook 35.

In [ ]:
def interaction(expression, z, genes):
    r = species_r(expression, z, pd.Index(genes))
    return (r.human - r.mouse).to_numpy()


p2_rows = []
for arm in INJECTED:
    genes = OWN_INJECTED_GENES[arm]
    cols_inj = genes_all.get_indexer(genes)
    counts = INJECTED[arm]
    injected_eval = log_normalize(counts[pt_rows], np.asarray(counts[pt_rows][:, measured_both].sum(axis=1)).ravel().astype(float))
    z0 = COORD[arm]['full']
    if arm == 'LRS-Z':
        ia, ib = INJ_HALVES[arm]
        z1 = lrs_zone(ia, ib, {**selA, 'human': INJ_SEL[arm]['A']}, {**selB, 'human': INJ_SEL[arm]['B']})
        z1[~human_pt] = z0[~human_pt]
    else:
        z1 = z0.copy()
        zz, _, _ = combined(arm, 'human', 'injection', sels=(INJ_SEL[arm]['A'], INJ_SEL[arm]['B']))
        z1[human_pt] = zz
    original = interaction(EVAL_ORIGINAL[:, cols_inj], z0, genes)
    frozen = interaction(injected_eval[:, cols_inj], z0, genes)
    refitted = interaction(injected_eval[:, cols_inj], z1, genes)
    moved = within_segment_agreement(z1, z0, specimen_pt, segment_pt)
    rec = {'arm': arm, **absorption_ratio(original, frozen, refitted, floor=.02),
           'human within-segment rho': float(moved[moved.specimen.isin(SAMPLES['human'])].rho.median())}
    rec['material absorption (notebook 35 flag)'] = bool(rec['median_ratio'] < .85 or rec['human within-segment rho'] < .90)
    p2_rows.append(rec)
p2_51 = pd.read_csv(upstream51 / 'p2_absorption.csv')
p2_51 = p2_51[p2_51.arm.isin(['LCP own landmarks', 'LRS own landmarks', 'SCF13', 'DPT13'])].assign(arm=lambda d: d.arm.str.replace(' own landmarks', '') + np.where(d.arm.str.startswith('LRS'), ' (notebook 51)', ''))
P2 = pd.concat([pd.DataFrame(p2_rows), p2_51], ignore_index=True)
P2.to_csv(output / 'p2_absorption.csv', index=False)
display(P2[['arm', 'median_ratio', 'slope', 'human within-segment rho', 'material absorption (notebook 35 flag)']])

### P4, P4c, P4c-dev (repaired) and WSR · held-out genes, cross-specimen within species

- **P4 and P4c** are unchanged from notebook 51.
- **P4c-dev** is repaired as protocol requires: genes with fewer than 10 training counts in a
  segment are excluded, and a ridge of 1 is used.
- **WSR** follows the protocol:
  - each specimen's residuals come from its own segment + coverage oracle;
  - per-gene, per-segment training splines are applied at test positions;
  - the statistic is the mean atanh(r) over all fold genes;
  - the null uses 500 within-segment permutations of the test coordinate, with seed 61 and the same
    permutations for every arm and fold.

Arms run in parallel processes.

In [ ]:
covariates = np.column_stack([log_library, np.log(neph_obs.n_spots.to_numpy(float)[pt_rows]),
                              np.log(np.asarray((COUNTS[pt_rows] > 0).sum(axis=1)).ravel())])
anatomy_code = pd.Series(segment_pt).map({'S1': 0, 'S2': 1, 'S3': 2}).to_numpy()
CELLS = []
RESID = {}
for sp, names in SAMPLES.items():
    genes_sp = evaluation_mask(sp, 'mouse_sn_male' if sp == 'mouse' else 'human_sn_halfB').loc[EVAL_GENES].to_numpy()
    for k in range(N_FOLDS):
        cols = genes_sp & (fold_of_eval == k)
        for n in names:
            m = specimen_pt == n
            y = EVAL_X[m][:, cols].toarray()
            own = fit_segment_covariate_oracle(y, covariates[m], anatomy_code[m], specimen_pt[m], degree=2)
            RESID[(sp, k, n)] = y - predict_segment_covariate_oracle(own, covariates[m], anatomy_code[m])
        count_cols = genes_all.get_indexer(EVAL_GENES[cols])
        for train, test in ((names[0], names[1]), (names[1], names[0])):
            tr, te = specimen_pt == train, specimen_pt == test
            ytr, yte = EVAL_X[tr][:, cols].toarray(), EVAL_X[te][:, cols].toarray()
            oracle = fit_segment_covariate_oracle(ytr, covariates[tr], anatomy_code[tr], specimen_pt[tr], degree=2)
            CELLS.append({'species': sp, 'fold': k, 'train': train, 'test': test, 'tr': tr, 'te': te, 'ytr': ytr, 'yte': yte,
                          'base_tr': predict_segment_covariate_oracle(oracle, covariates[tr], anatomy_code[tr]),
                          'base_te': predict_segment_covariate_oracle(oracle, covariates[te], anatomy_code[te]),
                          'ctr': PT_COUNTS[tr][:, count_cols].toarray(), 'cte': PT_COUNTS[te][:, count_cols].toarray()})
rng_wsr = np.random.default_rng(WSR_SEED)
PERMS = {}
for n in sorted(set(specimen_pt.astype(str))):
    for s in ('S1', 'S2', 'S3'):
        size = int(((specimen_pt == n) & (segment_pt == s)).sum())
        PERMS[(n, s)] = np.array([rng_wsr.permutation(size) for _ in range(WSR_PERMUTATIONS)])


def p4_arm(arm):
    out = []
    for cell in CELLS:
        z = COORD[arm][f"fold{cell['fold']}"]
        tr, te = cell['tr'], cell['te']
        p4 = heldout_prediction(cell['ytr'], z[tr], segment_pt[tr], cell['yte'], z[te], segment_pt[te], basis_df=BASIS_DF)
        spline = residual_spline_gain(cell['ytr'], z[tr], segment_pt[tr], cell['base_tr'], cell['yte'], z[te], segment_pt[te],
                                      cell['base_te'], basis_df=P4C_DF)
        dev = deviance_gain_repaired(cell['ctr'], LIBRARY_PT[tr], z[tr], segment_pt[tr], cell['cte'], LIBRARY_PT[te], z[te],
                                     segment_pt[te], basis_df=P4C_DF)
        for seg in ('all', 'S1', 'S2', 'S3'):
            r = np.ones(te.sum(), bool) if seg == 'all' else segment_pt[te] == seg
            var = np.maximum(p4['train_var'], 1e-12)
            out.append({'arm': arm, 'species': cell['species'], 'fold': cell['fold'], 'train': cell['train'], 'segment': seg,
                        'P4': (1 - np.sum(p4['coordinate_mse'] / var) / np.sum(p4['segment_mse'] / var)) if seg == 'all' else np.nan,
                        'P4c': scaled_gain(spline['base_mse'], spline['coordinate_mse'], spline['train_var'], r),
                        'P4c-dev (repaired)': float(1 - dev['coordinate_deviance'][r].sum() / dev['segment_deviance'][r].sum())})
    return out


def wsr_arm(arm):
    out = []
    for sp, names in SAMPLES.items():
        for train, test in ((names[0], names[1]), (names[1], names[0])):
            tr, te = specimen_pt == train, specimen_pt == test
            obs, null = {s: [] for s in ('S1', 'S2', 'S3')}, {s: [] for s in ('S1', 'S2', 'S3')}
            for k in range(N_FOLDS):
                z = COORD[arm][f'fold{k}']
                res = wsr_correlations(RESID[(sp, k, train)], z[tr], segment_pt[tr], RESID[(sp, k, test)], z[te], segment_pt[te],
                                       {s: PERMS[(test, s)] for s in ('S1', 'S2', 'S3')})
                for s, (o, nl) in res.items():
                    obs[s].append(o)
                    null[s].append(nl)
            for s in ('S1', 'S2', 'S3'):
                o, nl = np.concatenate(obs[s]), np.concatenate(null[s], axis=1)
                stat, null_stats = wsr_statistic(o), wsr_statistic(nl)
                out.append({'arm': arm, 'species': sp, 'segment': s, 'train': train, 'test': test, 'WSR': stat,
                            'null mean': float(np.mean(null_stats)), 'null max': float(np.max(null_stats)),
                            'p': float((1 + np.sum(null_stats >= stat)) / (1 + len(null_stats))), 'genes': int(len(o))})
    return out


def _limited(task):
    from threadpoolctl import threadpool_limits
    name, arm = task
    with threadpool_limits(1):
        return globals()[name](arm)


def _parallel(name, arms):
    with ProcessPoolExecutor(max_workers=min(WORKERS, len(arms)), mp_context=get_context('fork')) as pool:
        return [row for part in pool.map(_limited, [(name, a) for a in arms]) for row in part]


P4 = pd.DataFrame(_parallel('p4_arm', ARMS))
P4.to_csv(output / 'p4_p4c_heldout_prediction.csv', index=False)
WSR = pd.DataFrame(_parallel('wsr_arm', ARMS))
WSR.to_csv(output / 'wsr.csv', index=False)
saved_p4 = pd.read_csv(upstream36 / 'p4_heldout_prediction.csv')
check = P4[P4.segment.eq('all') & P4.arm.isin(['SCF13', 'DPT13'])].merge(saved_p4, on=['arm', 'species', 'fold', 'train'])
if len(check) != 24 or np.abs(check['P4'] - check['gain vs segment oracle']).max() > 1e-9:
    raise ValueError('SCF13/DPT13 P4 does not reproduce notebook 36.')
print('P4 for SCF13 and DPT13 reproduces notebook 36 (24 cells).')
display(P4.groupby(['species', 'segment', 'arm'])[['P4', 'P4c', 'P4c-dev (repaired)']].mean().unstack('arm').round(4))
display(WSR.pivot_table(index=['species', 'segment', 'test'], columns='arm', values='WSR').round(4))

## 10 · Read-split calibration of the ZAL posteriors, and cross-species curves

For ZAL, d = (m_AB − m_BA)/√(sd_AB² + sd_BA²). The two estimates come from disjoint read halves and
disjoint selections. The targets are as in notebook 52.

In [ ]:
calib_rows = []
for arm in ZAL_ARMS:
    for sp in SAMPLES:
        ab, ba = FITS[f'{arm}|{sp}|full|AB'], FITS[f'{arm}|{sp}|full|BA']
        d = standardized_difference(ab['mean'], ab['sd'], ba['mean'], ba['sd'])
        seg_of = segment_pt[species_pt == sp]
        for s in ('S1', 'S2', 'S3'):
            m = seg_of == s
            sdd, share = float(np.std(d[m])), float(np.mean(np.abs(d[m]) <= 1.2815516))
            calib_rows.append({'arm': arm, 'species': sp, 'segment': s, 'SD of d': sdd, 'share |d| <= 1.28': share,
                               'target met': bool(.85 <= sdd <= 1.15 and .75 <= share <= .85)})
for arm in ('LRS-Z', 'LRS-Z-PL'):
    d_all = standardized_difference(COORD[arm]['half_B'], LRSZ_SD[arm]['AB'], COORD[arm]['half_A'], LRSZ_SD[arm]['BA'])
    for sp in SAMPLES:
        d = d_all[species_pt == sp]
        seg_of = segment_pt[species_pt == sp]
        for s in ('S1', 'S2', 'S3'):
            m = seg_of == s
            sdd, share = float(np.std(d[m])), float(np.mean(np.abs(d[m]) <= 1.2815516))
            calib_rows.append({'arm': arm, 'species': sp, 'segment': s, 'SD of d': sdd, 'share |d| <= 1.28': share,
                               'target met': bool(.85 <= sdd <= 1.15 and .75 <= share <= .85)})
CALIB = pd.DataFrame(calib_rows)
CALIB.to_csv(output / 'calibration_read_split.csv', index=False)
if len(CALIB):
    display(CALIB.round(3))

# Cross-species within-zone curves of conserved held-out genes (C-reg).
conserved_eval = (contrast[['mouse_sn_male', 'human_sn_halfB']].abs().ge(CONSERVED_MIN).all(axis=1)
                  & (np.sign(contrast.mouse_sn_male) == np.sign(contrast.human_sn_halfB))
                  & detection.mouse.ge(MIN_DETECTION) & detection.human.ge(MIN_DETECTION)
                  & ~genes_all.isin(LANDMARKS['mouse'] + LANDMARKS['human'] + sum((landmarks_from(s) for s in SEL.values()), [])))
CURVE_GENES = genes_all[conserved_eval.to_numpy()]
curve_x = EVAL_ORIGINAL[:, genes_all.get_indexer(CURVE_GENES)]
xs = np.linspace(0, 1, CURVE_POINTS)


def zone_curves(z, sp, s, shift=0.0):
    from pseudospace.stats_gam import gam_internal_knots, make_gam_design
    m = (species_pt == sp) & (segment_pt == s)
    lo, hi = np.quantile(z[m], [.01, .99])
    x = np.clip((z[m] - lo) / (hi - lo), 0, 1)
    knots = gam_internal_knots(x, basis_df=4)
    D = make_gam_design(x, knots)
    coef = np.linalg.lstsq(D, curve_x[m].toarray(), rcond=None)[0]
    return make_gam_design(np.clip(xs + shift, 0, 1), knots) @ coef


curve_rows = []
for arm in ['SCF13', 'DPT13', 'LRS', 'LRS-Z'] + ZAL_ARMS:
    z = COORD[arm]['full']
    for s in ('S1', 'S2', 'S3'):
        mouse_c = zone_curves(z, 'mouse', s)
        for shift in (0.0, -CURVE_SHIFT, CURVE_SHIFT):
            human_c = zone_curves(z, 'human', s, shift)
            rho = [spearmanr(mouse_c[:, g], human_c[:, g]).statistic for g in range(len(CURVE_GENES))]
            curve_rows.append({'arm': arm, 'segment': s, 'human shift': shift, 'median Spearman': float(np.nanmedian(rho))})
CURVES = pd.DataFrame(curve_rows)
CURVES.to_csv(output / 'cross_species_curves.csv', index=False)
print(len(CURVE_GENES), 'conserved held-out genes')
display(CURVES.pivot_table(index=['arm', 'segment'], columns='human shift', values='median Spearman').round(3))

## 11 · Summaries, ZAL rules, adoption and claim gates (as pre-registered)

In [ ]:
METRICS = pd.DataFrame(rows)
METRICS.to_csv(output / 'metrics_long.csv', index=False)
cells = METRICS[METRICS.metric.isin(['gene-fold agreement', 'P5 equal depth', 'count split A vs B', 'position vs log library', 'LOSO'])]
by_segment = cells.groupby(['metric', 'species', 'segment', 'arm']).value.median().unstack('arm')
by_segment.to_csv(output / 'summary_cells_by_segment.csv')
p1w = METRICS[METRICS.metric.eq('P1w')]
p1w.groupby(['circular for landmark arms', 'species', 'segment', 'arm']).value.mean().unstack('arm').to_csv(output / 'summary_p1w.csv')
P4.groupby(['species', 'segment', 'arm'])[['P4', 'P4c', 'P4c-dev (repaired)']].mean().unstack('arm').to_csv(output / 'summary_p4.csv')
gaps = METRICS[METRICS.metric.str.startswith('registration') & METRICS.refit.ne('full')]
display(by_segment.round(3))


def fold_summary(arm, sp, seg=None):
    t = METRICS[METRICS.metric.eq('gene-fold agreement') & METRICS.arm.eq(arm) & METRICS.species.eq(sp)]
    per = t.groupby('segment').value.median()
    return float(per.loc[seg]) if seg else float(per.mean())


def summary_row(arm):
    out = {}
    for sp in SAMPLES:
        out[f'gene-fold {sp}'] = fold_summary(arm, sp)
        out[f'P1w {sp}'] = float(p1w[p1w.arm.eq(arm) & p1w.species.eq(sp) & ~p1w['circular for landmark arms']].value.mean())
        ref = 'mouse_sn_male' if sp == 'mouse' else 'human_sn_halfB'
        out[f'P1 {sp}'] = float(METRICS[METRICS.metric.eq('P1') & METRICS.arm.eq(arm) & METRICS.reference.eq(ref)].value.iloc[0])
        cellsp = P4[P4.arm.eq(arm) & P4.species.eq(sp) & P4.segment.eq('all')]
        out[f'P4c mean {sp}'] = float(cellsp.P4c.mean())
        out[f'P4c > 0 cells {sp}'] = int((cellsp.P4c > 0).sum())
        out[f'P4 mean {sp}'] = float(cellsp.P4.mean())
    out['human S1 gene-fold'] = fold_summary(arm, 'human', 'S1')
    p5 = METRICS[METRICS.metric.eq('P5 equal depth') & METRICS.arm.eq(arm)]
    out['P5 overall'] = float(p5.value.median())
    out['P5 human S1'] = float(p5[p5.species.eq('human') & p5.segment.eq('S1')].value.mean())
    out['registration gap'] = float(gaps[gaps.arm.eq(arm)].value.abs().max())
    lo = METRICS[METRICS.metric.eq('LOSO') & METRICS.arm.eq(arm)]
    out['LOSO'] = float(lo.value.median()) if len(lo) else np.nan
    out['WSR mean'] = float(WSR[WSR.arm.eq(arm)].WSR.mean())
    return out


SUMMARY = pd.DataFrame({arm: summary_row(arm) for arm in ARMS})
SUMMARY.to_csv(output / 'summary_by_arm.csv')
display(SUMMARY.round(4))
S = SUMMARY

# ZAL keep/abandon rules against LCP-ext (plan §2), and the same comparison for LRS-Z against LRS-ext.
ZAL_MAIN = 'ZAL' if MODEL else 'LRS-Z'
dgf = S.loc['gene-fold mouse', ZAL_MAIN] - S.loc['gene-fold mouse', 'LCP']
dp1w = S.loc['P1w mouse', ZAL_MAIN] - S.loc['P1w mouse', 'LCP']
ZAL_RULES = {'main zone arm': ZAL_MAIN, 'gauge note': 'zone units: the registration gap is near-tautological (flagged)', 'mouse gene-fold minus LCP-ext': float(dgf), 'mouse P1w minus LCP-ext': float(dp1w),
             'keep (gene-fold >= -0.03 and |P1w| within 0.03)': bool(dgf >= -.03 and abs(dp1w) <= .03),
             'abandon (gene-fold drop > 0.10 or P1w drop > 0.05)': bool(dgf < -.10 or dp1w < -.05),
             'LRS-Z minus LRS-ext, mouse gene-fold': float(S.loc['gene-fold mouse', 'LRS-Z'] - S.loc['gene-fold mouse', 'LRS']),
             'LRS-Z minus LRS-ext, mouse P1w': float(S.loc['P1w mouse', 'LRS-Z'] - S.loc['P1w mouse', 'LRS'])}
main_calib = CALIB[CALIB.arm.eq('ZAL' if MODEL else 'LRS-Z')]
CALIBRATION_ABANDONED = bool((~main_calib['target met']).sum() > len(main_calib) / 2) if len(main_calib) else None
adopted_arm = ZAL_MAIN


def adoption(arm):
    a = S[arm]
    p2 = P2.set_index('arm')
    absorbed = bool(p2.loc[arm, 'material absorption (notebook 35 flag)']) if arm in p2.index and pd.notna(
        p2.loc[arm].get('material absorption (notebook 35 flag)', np.nan)) else False
    return {'(A) P1 >= SCF13 - 0.03, both species': bool(all(a[f'P1 {sp}'] >= S.loc[f'P1 {sp}', 'SCF13'] - .03 for sp in SAMPLES)),
            '(A) P5 >= 0.80 overall and >= 0.70 human S1': bool(a['P5 overall'] >= .80 and a['P5 human S1'] >= .70),
            '(A) registration gap <= 0.15': bool(a['registration gap'] <= .15),
            '(A) no material P2 absorption': not absorbed,
            '(B) P1w >= DPT13 + 0.05 mouse, >= DPT13 human': bool(a['P1w mouse'] >= S.loc['P1w mouse', 'DPT13'] + .05
                                                                  and a['P1w human'] >= S.loc['P1w human', 'DPT13']),
            '(B) gene-fold >= 0.70 mouse, >= 0.59 human': bool(a['gene-fold mouse'] >= .70 and a['gene-fold human'] >= .59),
            '(B) LOSO >= 0.85 (n/a counts as pass for the per-structure ratio)': bool(np.isnan(a['LOSO']) or a['LOSO'] >= .85)}


ADOPTION = {arm: adoption(arm) for arm in ARMS}
pd.DataFrame(ADOPTION).to_csv(output / 'adoption_criteria.csv')

# Claim gates (reported for every arm).
wsr = WSR.copy()
wsr['sig'] = wsr.p <= WSR_ALPHA
cellsig = wsr.groupby(['arm', 'species', 'segment']).sig.all()
cellstat = wsr.groupby(['arm', 'species', 'segment', 'test']).WSR.first()
claim_rows = []
for arm in ARMS:
    s = S[arm]
    p4c_claim = bool(s['P4c > 0 cells mouse'] >= 5 and s['P4c > 0 cells human'] >= 4 and all(
        s[f'P4c mean {sp}'] > max(S.loc[f'P4c mean {sp}', x] for x in ('SCF13', 'DPT13', 'LRS') if x != arm) for sp in SAMPLES))
    for sp in SAMPLES:
        for seg in ('S1', 'S2', 'S3'):
            passes = bool(cellsig.loc[(arm, sp, seg)])
            beats_placebo = bool(all(cellstat.loc[(arm, sp, seg, t)] > cellstat.loc[(PLACEBO_ARM, sp, seg, t)] for t in SAMPLES[sp]))
            placebo_fails = not bool(cellsig.loc[(PLACEBO_ARM, sp, seg)])
            claim_rows.append({'arm': arm, 'species': sp, 'segment': seg, 'WSR passes (p <= 0.01 both directions)': passes,
                               'beats placebo both directions': beats_placebo, 'placebo fails': placebo_fails,
                               'within-segment information claim': passes and beats_placebo and placebo_fails,
                               'held-out prediction claim (P4c)': p4c_claim})
CLAIMS = pd.DataFrame(claim_rows)
CLAIMS.to_csv(output / 'claim_gates.csv', index=False)
HUMAN_S1_UNRESOLVED = bool(S.loc['human S1 gene-fold', adopted_arm] < .50 or S.loc['P5 human S1', adopted_arm] < .70)
DECISION = {'notebook 52 adopted configuration': ADOPTED, 'zone arms fitted as models': MODEL, 'ZAL rules': ZAL_RULES,
            'calibration abandoned (ZAL read split)': CALIBRATION_ABANDONED,
            'convergence fix failed (> 10% of fits at the cap)': bool(CAP_SHARE > .10) if np.isfinite(CAP_SHARE) else None,
            'share of fits at the cap': CAP_SHARE, 'adoption criteria for the main zone arm': ADOPTION[adopted_arm],
            'main zone arm passes (A) without the downstream gate and (B) without P4c': all(ADOPTION[adopted_arm].values()),
            'human S1 unresolved': HUMAN_S1_UNRESOLVED, 'temperatures': TEMPERATURE}
print(json.dumps(DECISION, indent=1, default=str))
display(CLAIMS.pivot_table(index=['species', 'segment'], columns='arm', values='within-segment information claim'))

## 12 · Coordinate files for workstream B and the record

Every arm is written in `primary_pt_coordinate.csv` format. Model arms add posterior columns, and
LRS-U adds its bootstrap SD. Workstream B's notebooks run on the surviving arms and on the placebo.

In [ ]:
coord_dir = output_root / 'coordinates'
coord_dir.mkdir(exist_ok=True)
frame = pd.DataFrame({'structure_id': saved_pt.index, 'specimen': specimen_pt, 'species': species_pt,
                      'segment_class': saved_pt.segment_class.astype(str).to_numpy(), 'position': COORD['LRS']['full'],
                      'coordinate_method': 'LRS', 'method_description': 'external-landmark ratio, within-species rank'})
for k in range(N_FOLDS):
    frame[f'fold_{k}'] = COORD['LRS'][f'fold{k}']
frame.to_csv(coord_dir / 'lrs_ext.csv', index=False)
(coord_dir / 'lrs_ext_construction_genes.txt').write_text('\n'.join(sorted(set(LANDMARKS['mouse']) | set(LANDMARKS['human']))) + '\n')
for arm in ['LRS-Z', 'LRS-Z-PL'] + ZAL_ARMS:
    frame = pd.DataFrame({'structure_id': saved_pt.index, 'specimen': specimen_pt, 'species': species_pt,
                          'segment_class': saved_pt.segment_class.astype(str).to_numpy(), 'position': COORD[arm]['full'],
                          'coordinate_method': arm, 'method_description': f'notebook 53 arm {arm} (zone units / 3)'})
    if arm in SD:
        frame['posterior_sd'] = SD[arm]['full']
    if arm in LRSZ_SD:
        frame['position_sd'] = np.sqrt(LRSZ_SD[arm]['AB'] ** 2 + LRSZ_SD[arm]['BA'] ** 2) / 2
    for k in range(N_FOLDS):
        frame[f'fold_{k}'] = COORD[arm][f'fold{k}']
    frame.to_csv(coord_dir / f"{arm.lower().replace('-', '_')}.csv", index=False)
    placebo = arm.endswith('PL')
    construction = sorted(set().union(*[landmarks_from(SEL[(sp, h)], None, placebo) for sp in SAMPLES for h in 'AB']))
    (coord_dir / f"{arm.lower().replace('-', '_')}_construction_genes.txt").write_text('\n'.join(construction) + '\n')
with open(output / 'summary.json', 'w') as handle:
    json.dump({'logic_version': NOTEBOOK_LOGIC_VERSION, 'decision': DECISION, 'summary_by_arm': S.round(6).to_dict(),
               'curves': CURVES[CURVES['human shift'].eq(0)].groupby('arm')['median Spearman'].mean().to_dict(),
               'n_fits': len(FIT_DIAG)}, handle, indent=1, default=str)
print('Saved coordinates to', coord_dir)